# Training Results Viewer

This notebook allows you to load and visualize the results of training an autoencoder on the EEG dataset. You can specify which fold's results to load and then view the reconstruction performance of the model on the test set.

Requirements:
 - Model checkpoints at path: core.settings.FINETUNE_OUTPUT_DIR/your_model_name
 - Preprocessed data at path: core.settings.FINETUNE_DIR

*To obtain this data, please run make_dataset.py and run_general_finetuning.py beforehand*
### Specify parameters

In [1]:

import json

# specify which fold to load (0-5)
fold_to_load = 4
model_name = "test4"
epochs_to_load = [0]
load_best = True

In [2]:
import core
import torch
from core.architectures import EEGAutoencoder
from core.datasets import KFoldSplitter
import core.settings as settings
from pathlib import Path
import matplotlib.pyplot as plt
import os
from libs.labram import modeling_finetune, utils
from einops import rearrange
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact


if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

splitter = KFoldSplitter(k_folds=6)

folds = splitter.split_by_subject()


fold = folds[fold_to_load]
tensors = fold.get_datasets()
train_dataset = tensors["train"]
val_dataset = tensors["val"]
test_dataset = tensors["test"]

slider = widgets.IntSlider(
        min=0,
        max=len(test_dataset)-1,
        step=1,
        value=0,
        description=f"Sample: ",
        layout = widgets.Layout(width="100%", align="center"),
    )

save_button = widgets.Button(
    description="Save Figure",
    layout=widgets.Layout(width="100%", align="center"),
)
save_message = widgets.Output()

ch_names = settings.CHORDER_STANDARD
# cut off "-REF" and "EEG " from channel names for compatibility with get_input_chans function
ch_names = [name.split(' ')[-1].split('-')[0] for name in ch_names]
input_chans = utils.get_input_chans(ch_names)


fold_path = settings.FINETUNE_OUTPUT_DIR / model_name / Path(str(fold_to_load + 1))
metadata_path = settings.FINETUNE_OUTPUT_DIR / model_name

checkpoint_paths = [fold_path / f"checkpoint-{epoch}.pth" for epoch in epochs_to_load]

if load_best:
    checkpoint_paths.append(fold_path / "checkpoint-best_test.pth")

with open(metadata_path / 'metadata.json', 'r') as f:
    metadata = json.load(f)

params = metadata["hyperparameters"]

model = EEGAutoencoder(train_dataset.target_dim).eval()

def show_reconstructions(sample_idx, save= True):
    plt.figure(figsize=(12, 4*len(checkpoint_paths)))
    param_text = f"LR: {params['lr']} | WD: {params['weight_decay']}"
    plt.suptitle(f"Autoencoder reconstruction plots\n{param_text}", y=1, fontweight="bold")

    epoch = test_dataset[sample_idx][0]
    test_input = epoch.unsqueeze(0)
    device = next(model.parameters()).device
    test_input = rearrange(test_input, 'B N (A T) -> B N A T', T=200)
    test_input = test_input.float().to(device, non_blocking=True) / 100
    true_wave = epoch[0, :].cpu().numpy()
    true_wave_avg = np.mean(epoch.cpu().numpy(), axis=0)

    for checkpoint_path in checkpoint_paths:
        if checkpoint_path.exists():
            checkpoint = torch.load(checkpoint_path)
            model.load_state_dict(checkpoint["model"])
            with torch.no_grad():
                reconstruction = model(test_input, input_chans).squeeze(0)
            epoch_idx = checkpoint["epoch"] if checkpoint["epoch"] != "best_test" else str(metadata['results']['max_r2_test_epochs'][fold_to_load])+' (Best test results)'


            reconstruction = reconstruction.unflatten(dim=0, sizes=(23, 200))
            recon_wave = reconstruction[0, :].cpu().numpy()

            # Plot the true and reconstructed waveforms
            i = checkpoint_paths.index(checkpoint_path)
            plt.subplot(len(checkpoint_paths), 1, i + 1)
            time = np.arange(len(true_wave)) / settings.RSFREQ * 1000
            plt.plot(time,true_wave, label='True', alpha=0.7)
            plt.plot(time, recon_wave, label='Reconstructed', linestyle='--', alpha=0.9)
            plt.title(f"Fold {fold_to_load}, test sample {sample_idx} - Epoch: {epoch_idx}", loc='left')
            plt.xlabel("Time (ms)")
            plt.ylabel("Amplitude [uV]")
            plt.legend()
            plt.grid(True, alpha=0.3)


    plt.tight_layout()
    if save:
        save_dir = settings.PLOT_DIR / model_name / "reconstructions"
        save_dir.mkdir(parents=True, exist_ok=True)
        filename = save_dir / f"sample_{sample_idx}_reconstruction.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print("ran")
    plt.show()


save_button.on_click(show_reconstructions)

interact(
    show_reconstructions,
    sample_idx= slider,
    save=widgets.fixed(False)
)

display(save_button, save_message)
def on_save_clicked(b):
    show_reconstructions(slider.value, save=True)

save_button.on_click(on_save_clicked)

interactive(children=(IntSlider(value=0, description='Sample: ', layout=Layout(width='100%'), max=539), Output…

Button(description='Save Figure', layout=Layout(width='100%'), style=ButtonStyle())

Output()